In [7]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from transformers import AutoImageProcessor, AutoModelForImageClassification
from torch import nn, optim

In [ ]:
# Transformações básicas (ajustáveis depois)
transform = transforms.Compose([
    transforms.Resize((224, 224)),   
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])  
])

dataset = datasets.ImageFolder(root="data", transform=transform)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"Treino: {len(train_dataset)} imagens | Validação: {len(val_dataset)} imagens")


Treino: 1600 imagens | Validação: 400 imagens


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint = "google/vit-base-patch16-224"
model = AutoModelForImageClassification.from_pretrained(
    checkpoint,
    num_labels=2,
    ignore_mismatched_sizes=True   
).to(device)

processor = AutoImageProcessor.from_pretrained(checkpoint)


Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([2]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 1405.60it/s]
Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.
Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 4718.00it/s]


In [10]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

def train_one_epoch(model, dataloader):
    model.train()
    total_loss = 0
    for imgs, labels in dataloader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs).logits
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

def evaluate(model, dataloader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in dataloader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs).logits
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total


In [11]:
for epoch in range(3):  # artigo usou 10, mas testamos com 3
    loss = train_one_epoch(model, train_loader)
    y_pred, y_true, y_prob = evaluate(model, val_loader)

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)

    print(f"Época {epoch+1}: Loss={loss:.4f} | Acc={acc:.4f} | Prec={prec:.4f} | "
          f"Rec={rec:.4f} | F1={f1:.4f} | AUC={auc:.4f}")

KeyboardInterrupt: 